# Training MobileNetV2 - KucingVision.AI
Jalankan cell satu per satu dari atas ke bawah.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Sesuaikan path dataset di Google Drive kamu
# Contoh: jika folder dataset ada di 'My Drive/dataset_kucing/'
DATASET_DIR = '/content/drive/MyDrive/dataset_kucing'  # <-- GANTI INI
MODEL_OUTPUT = '/content/drive/MyDrive/model.h5'       # model disimpan ke Drive

import os
print('Train classes:', os.listdir(os.path.join(DATASET_DIR, 'train')))
print('Val classes  :', os.listdir(os.path.join(DATASET_DIR, 'val')))

In [ ]:
# 3. Import library
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print('TensorFlow version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 4. Konfigurasi
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 30
LR          = 1e-4
NUM_CLASSES = 5  # Healthy, Ringworm, Scabies, Flea Allergy, dermatitis

In [ ]:
# 5. Load dataset
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, 'train'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

# Simpan urutan kelas (penting!)
CLASS_NAMES = train_ds.class_names
print('Urutan kelas:', CLASS_NAMES)

# Augmentasi + preprocessing
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])

train_ds = train_ds.map(lambda x, y: (augmentation(x, training=True), y),
                        num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.map(lambda x, y: (preprocess(x), y),
                        num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (preprocess(x), y),
                      num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 6. Build model
base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)
model.compile(optimizer=Adam(LR), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 7. Fase 1: Train head layer (10 epoch)
callbacks = [
    ModelCheckpoint(MODEL_OUTPUT, save_best_only=True, monitor='val_accuracy', verbose=1),
    EarlyStopping(patience=7, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss', verbose=1),
]

print('=== FASE 1: Training head layer ===')
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
# 8. Fase 2: Fine-tune 50 layer terakhir
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

model.compile(optimizer=Adam(LR / 10), loss='categorical_crossentropy', metrics=['accuracy'])

print('=== FASE 2: Fine-tuning ===')
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)

In [ ]:
# 9. Cek urutan kelas final (PENTING - samakan dengan CLASS_LABELS di app.py)
print('Urutan kelas model (sesuaikan CLASS_LABELS di app.py):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i}: {name}')

print(f'\nModel tersimpan di: {MODEL_OUTPUT}')